<a href="https://colab.research.google.com/github/yousefwerida28/Flyrank-ML-intern/blob/main/capstone_archetype_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SETUP**

In [ ]:
from google.colab import userdata
import duckdb, os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

months = sorted(set(
    f.split("/")[1].replace("month=", "")
    for f in files
    if f.startswith("fact_content_daily_performance/month=")
))

print(months)

['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


**Step 2**

In [ ]:
monthly_summary = con.sql("""
    SELECT
        content_hash_id,
        strftime(report_date, '%Y-%m') AS month,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet'
    ])
    GROUP BY content_hash_id, strftime(report_date, '%Y-%m')
""").df()

print(monthly_summary.shape)
monthly_summary.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2075497, 5)


,content_hash_id,month,impressions,clicks,avg_position
0,content_65b8a610174a1036,2026-01,2207.0,22.0,5.505318
1,content_3fc85bdef381af2d,2026-01,2259.0,0.0,1.507169
2,content_80071808216aef39,2026-01,52196.0,90.0,4.923048
3,content_fd2a7e03887e7881,2026-01,10519.0,36.0,2.827487
4,content_19f818ed91e960ca,2026-01,1363.0,5.0,3.342283


**Step 3**

In [ ]:
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

In [ ]:
test = con.execute("""
SELECT COUNT(*) AS row_count
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").df()

display(test)

,row_count
0,9841378


In [ ]:
query = """
WITH daily AS (
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-10/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-11/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ])
),

monthly AS (
    SELECT
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr,

        AVG(gsc_avg_position) AS avg_position

    FROM daily
    GROUP BY
        content_hash_id,
        DATE_TRUNC('month', report_date)
),

features AS (
    SELECT
        content_hash_id,

        -- Performance level
        AVG(impressions) AS avg_monthly_impressions,
        AVG(clicks) AS avg_monthly_clicks,
        AVG(ctr) AS avg_ctr,
        AVG(avg_position) AS avg_position,

        -- Overall volume
        SUM(impressions) AS total_impressions,
        SUM(clicks) AS total_clicks,

        -- Trend over the six months
        REGR_SLOPE(impressions, EPOCH(month)) AS impression_trend,
        REGR_SLOPE(clicks, EPOCH(month)) AS click_trend,
        REGR_SLOPE(ctr, EPOCH(month)) AS ctr_trend,
        REGR_SLOPE(avg_position, EPOCH(month)) AS position_trend,

        -- Consistency
        STDDEV(impressions) AS impressions_std,
        STDDEV(ctr) AS ctr_std,
        STDDEV(avg_position) AS position_std,

        -- Data coverage
        COUNT(*) AS months_observed

    FROM monthly
    GROUP BY content_hash_id
)

SELECT *
FROM features
WHERE months_observed >= 4
ORDER BY total_impressions DESC
LIMIT 1000
"""

features_df = con.execute(query).df()

print("Shape:", features_df.shape)

print("\nColumn names:")
print(features_df.columns.tolist())

print("\nSample rows:")
display(features_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (1000, 15)

Column names:
['content_hash_id', 'avg_monthly_impressions', 'avg_monthly_clicks', 'avg_ctr', 'avg_position', 'total_impressions', 'total_clicks', 'impression_trend', 'click_trend', 'ctr_trend', 'position_trend', 'impressions_std', 'ctr_std', 'position_std', 'months_observed']

Sample rows:


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,avg_position,total_impressions,total_clicks,impression_trend,click_trend,ctr_trend,position_trend,impressions_std,ctr_std,position_std,months_observed
0,content_eadb33b5df496f4a,152671.666667,1553.166667,0.010329,2.960705,916030.0,9319.0,0.035989,0.000331,2.990746e-10,-1.484763e-07,231489.516166,0.005882,0.656855,6
1,content_e241d6415ac9e534,144226.500000,381.666667,0.002669,3.518653,865359.0,2290.0,0.003472,-0.000001,-7.726539e-11,-7.765040e-08,34476.574231,0.000540,0.419514,6
2,content_e8a52cf3d5988c07,134836.833333,606.500000,0.004983,11.916351,809021.0,3639.0,0.009615,-0.000006,-3.486375e-10,2.807966e-07,60531.390192,0.001800,2.025773,6
3,content_e7b5dd4dff461ad2,123138.000000,1433.333333,0.010949,5.886400,738828.0,8600.0,0.011292,0.000174,4.197609e-10,-2.405084e-07,56258.001241,0.002856,1.250176,6
4,content_6302b8bce0bb84cb,111285.500000,727.666667,0.006433,2.678569,667713.0,4366.0,0.001934,0.000002,-9.840419e-11,-3.151943e-08,17511.489540,0.001782,0.544702,6
5,content_4d0d79fc12632ef8,110060.666667,560.166667,0.005326,4.506422,660364.0,3361.0,0.002270,0.000009,-4.129202e-11,-1.062345e-07,55782.699224,0.000815,0.915746,6
6,content_cf651123f1085418,108313.500000,223.666667,0.002039,6.572527,649881.0,1342.0,0.002540,0.000006,1.005446e-11,-1.082920e-07,21730.830824,0.000280,0.639273,6
7,content_512dbad65bd5ade9,100023.000000,1888.166667,0.020720,3.462386,600138.0,11329.0,0.013807,0.000242,-7.052375e-10,-1.289109e-07,73935.193690,0.004335,0.914522,6
8,content_ec2e0346994fb5a5,97355.333333,473.166667,0.003824,2.531714,584132.0,2839.0,0.017540,0.000096,2.974923e-10,3.793378e-08,91521.280558,0.002043,0.270095,6
9,content_b17c1d1cb0a346d6,96035.666667,417.833333,0.004328,5.120176,576214.0,2507.0,0.001250,-0.000007,-1.093029e-10,-6.502784e-08,21308.167774,0.001065,0.463718,6


In [ ]:
query = """
WITH daily AS (
    SELECT
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM read_parquet([
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-10/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-11/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-12/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet',
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    ])
),

monthly AS (
    SELECT
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr,

        AVG(gsc_avg_position) AS avg_position

    FROM daily
    GROUP BY
        content_hash_id,
        DATE_TRUNC('month', report_date)
),

monthly_with_index AS (
    SELECT
        *,
        DATE_DIFF(
            'month',
            DATE '2025-10-01',
            month
        ) AS month_number
    FROM monthly
),

features AS (
    SELECT
        content_hash_id,

        -- Performance level
        AVG(impressions) AS avg_monthly_impressions,
        AVG(clicks) AS avg_monthly_clicks,
        AVG(ctr) AS avg_ctr,
        AVG(avg_position) AS avg_position,

        -- Trend per month
        REGR_SLOPE(impressions, month_number) AS impression_trend,
        REGR_SLOPE(clicks, month_number) AS click_trend,
        REGR_SLOPE(ctr, month_number) AS ctr_trend,
        REGR_SLOPE(avg_position, month_number) AS position_trend,

        -- Consistency / volatility
        STDDEV(impressions) AS impressions_std,
        STDDEV(ctr) AS ctr_std,
        STDDEV(avg_position) AS position_std,

        -- Used only for data-quality filtering
        COUNT(*) AS months_observed

    FROM monthly_with_index
    GROUP BY content_hash_id
)

SELECT
    content_hash_id,
    avg_monthly_impressions,
    avg_monthly_clicks,
    avg_ctr,
    avg_position,
    impression_trend,
    click_trend,
    ctr_trend,
    position_trend,
    impressions_std,
    ctr_std,
    position_std
FROM features
WHERE months_observed >= 4

"""

features_df = con.execute(query).df()

print("Shape:", features_df.shape)

print("\nColumn names:")
print(features_df.columns.tolist())

print("\nSample rows:")
display(features_df.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (254802, 12)

Column names:
['content_hash_id', 'avg_monthly_impressions', 'avg_monthly_clicks', 'avg_ctr', 'avg_position', 'impression_trend', 'click_trend', 'ctr_trend', 'position_trend', 'impressions_std', 'ctr_std', 'position_std']

Sample rows:


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,avg_position,impression_trend,click_trend,ctr_trend,position_trend,impressions_std,ctr_std,position_std
0,content_c2b73a66f34d33f7,41.166667,0.166667,0.000926,4.859292,29.400000,0.142857,0.000794,1.471285,71.661473,0.002268,6.463573
1,content_e1abc61c37c2f207,78.333333,0.833333,0.014963,21.020706,32.457143,0.257143,0.003614,-2.851720,83.154475,0.028620,18.962225
2,content_e4b7f931fdfbacb5,322.166667,0.833333,0.002373,11.263688,-9.857143,-0.428571,-0.001391,1.122907,95.386407,0.003679,3.510226
3,content_78aa8c302b7a93c6,11.166667,0.000000,0.000000,35.017507,3.457143,0.000000,0.000000,-7.816287,7.440878,0.000000,21.153957
4,content_19af0b794a200e2c,320.000000,0.666667,0.002251,12.383231,83.257143,-0.057143,-0.000916,-1.560158,185.337530,0.003734,5.099990
5,content_4823ba07c84f44b1,130.000000,0.000000,0.000000,62.550529,31.771429,0.000000,0.000000,-0.868882,78.656214,0.000000,14.059381
6,content_00bce589338bfdce,561.666667,6.166667,0.017222,8.410569,187.142857,0.485714,-0.005806,-0.461512,404.986749,0.011635,1.387682
7,content_820631d266c8322b,1701.000000,1.833333,0.002907,7.131386,822.571429,-0.542857,-0.001345,-1.059524,1772.049774,0.002610,2.304595
8,content_7512edab490dfbb4,2273.166667,23.833333,0.010837,3.954973,321.114286,2.428571,-0.000419,-0.365312,874.660487,0.002517,0.831252
9,content_e43c2dbff9662740,135.333333,0.333333,0.007733,49.297080,37.257143,-0.171429,-0.005721,3.781657,103.602445,0.015385,21.379734


***STEP 4***

In [ ]:
# Look at the distribution of the main baseline features

baseline_features = [
    "avg_monthly_impressions",
    "avg_monthly_clicks",
    "avg_ctr",
    "avg_position",
    "impression_trend",
    "click_trend",
    "ctr_trend",
    "position_trend"
]

print("Feature quantiles:\n")

display(
    features_df[baseline_features].quantile(
        [0.25, 0.50, 0.75]
    )
)

Feature quantiles:



,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,avg_position,impression_trend,click_trend,ctr_trend,position_trend
0.25,17868.125000,36.958333,0.001823,3.641200,1752.307143,1.164286,-0.000427,-0.358926
0.50,22395.166667,69.166667,0.003171,4.778092,4476.400000,8.128571,-0.000073,-0.115508
0.75,30730.916667,134.583333,0.004966,6.329254,8778.460714,21.000000,0.000174,0.235731


In [ ]:
# Recalculate baseline thresholds using the full eligible population

thresholds = {
    "high_impressions": features_df["avg_monthly_impressions"].quantile(0.75),
    "low_impressions": features_df["avg_monthly_impressions"].quantile(0.25),

    "high_clicks": features_df["avg_monthly_clicks"].quantile(0.75),
    "low_clicks": features_df["avg_monthly_clicks"].quantile(0.25),

    "high_ctr": features_df["avg_ctr"].quantile(0.75),
    "low_ctr": features_df["avg_ctr"].quantile(0.25),

    "good_position": features_df["avg_position"].quantile(0.25),
    "weak_position": features_df["avg_position"].quantile(0.75)
}

print("Baseline thresholds:\n")

for name, value in thresholds.items():
    print(f"{name}: {value:.6f}")

Baseline thresholds:

high_impressions: 106.833333
low_impressions: 0.000000
high_clicks: 0.166667
low_clicks: 0.000000
high_ctr: 0.000656
low_ctr: 0.000000
good_position: 5.666667
weak_position: 19.222420


In [ ]:
import numpy as np

# Start fresh from the full feature dataset
baseline = features_df.copy()

# --------------------------------------------------
# 1. Performance signals
# --------------------------------------------------

baseline["strong_impressions"] = (
    baseline["avg_monthly_impressions"] >= thresholds["high_impressions"]
)

baseline["strong_clicks"] = (
    baseline["avg_monthly_clicks"] >= thresholds["high_clicks"]
)

baseline["strong_ctr"] = (
    baseline["avg_ctr"] >= thresholds["high_ctr"]
)

baseline["strong_position"] = (
    baseline["avg_position"] <= thresholds["good_position"]
)

baseline["weak_impressions"] = (
    baseline["avg_monthly_impressions"] <= thresholds["low_impressions"]
)

baseline["weak_clicks"] = (
    baseline["avg_monthly_clicks"] <= thresholds["low_clicks"]
)

baseline["weak_ctr"] = (
    baseline["avg_ctr"] <= thresholds["low_ctr"]
)

baseline["weak_position"] = (
    baseline["avg_position"] >= thresholds["weak_position"]
)


# Count strong and weak signals
baseline["strong_signal_count"] = baseline[
    [
        "strong_impressions",
        "strong_clicks",
        "strong_ctr",
        "strong_position"
    ]
].sum(axis=1)

baseline["weak_signal_count"] = baseline[
    [
        "weak_impressions",
        "weak_clicks",
        "weak_ctr",
        "weak_position"
    ]
].sum(axis=1)


# Performance category
baseline["performance_level"] = np.select(
    [
        baseline["strong_signal_count"] >= 2,
        baseline["weak_signal_count"] >= 2
    ],
    [
        "strong",
        "weak"
    ],
    default="middle"
)


# --------------------------------------------------
# 2. Trend signals
# --------------------------------------------------

baseline["impressions_improving"] = baseline["impression_trend"] > 0
baseline["clicks_improving"] = baseline["click_trend"] > 0
baseline["ctr_improving"] = baseline["ctr_trend"] > 0

# Lower position = better ranking
baseline["position_improving"] = baseline["position_trend"] < 0


baseline["improving_signal_count"] = baseline[
    [
        "impressions_improving",
        "clicks_improving",
        "ctr_improving",
        "position_improving"
    ]
].sum(axis=1)

baseline["declining_signal_count"] = (
    4 - baseline["improving_signal_count"]
)


# Trend category
baseline["trend_level"] = np.select(
    [
        baseline["improving_signal_count"] >= 3,
        baseline["declining_signal_count"] >= 3
    ],
    [
        "improving",
        "declining"
    ],
    default="mixed_stable"
)


# --------------------------------------------------
# 3. Final baseline archetype
# --------------------------------------------------

baseline["baseline_archetype"] = (
    baseline["performance_level"]
    + "_"
    + baseline["trend_level"]
)


print("Baseline shape:", baseline.shape)

display(
    baseline[
        [
            "content_hash_id",
            "performance_level",
            "trend_level",
            "baseline_archetype"
        ]
    ].head(10)
)

Baseline shape: (254802, 31)


,content_hash_id,performance_level,trend_level,baseline_archetype
0,content_c2b73a66f34d33f7,strong,improving,strong_improving
1,content_e1abc61c37c2f207,strong,improving,strong_improving
2,content_e4b7f931fdfbacb5,strong,declining,strong_declining
3,content_78aa8c302b7a93c6,weak,mixed_stable,weak_mixed_stable
4,content_19af0b794a200e2c,strong,mixed_stable,strong_mixed_stable
5,content_4823ba07c84f44b1,weak,mixed_stable,weak_mixed_stable
6,content_00bce589338bfdce,strong,improving,strong_improving
7,content_820631d266c8322b,strong,mixed_stable,strong_mixed_stable
8,content_7512edab490dfbb4,strong,improving,strong_improving
9,content_e43c2dbff9662740,strong,declining,strong_declining


In [ ]:
# --------------------------------------------------
# Correct trend classification
# --------------------------------------------------

# Positive trend = improving
# Negative trend = declining
# Zero trend = neutral

baseline["improving_signal_count"] = (
    (baseline["impression_trend"] > 0).astype(int)
    + (baseline["click_trend"] > 0).astype(int)
    + (baseline["ctr_trend"] > 0).astype(int)
    + (baseline["position_trend"] < 0).astype(int)
)

baseline["declining_signal_count"] = (
    (baseline["impression_trend"] < 0).astype(int)
    + (baseline["click_trend"] < 0).astype(int)
    + (baseline["ctr_trend"] < 0).astype(int)
    + (baseline["position_trend"] > 0).astype(int)
)


# Assign trend category
baseline["trend_level"] = np.select(
    [
        baseline["improving_signal_count"] >= 3,
        baseline["declining_signal_count"] >= 3
    ],
    [
        "improving",
        "declining"
    ],
    default="mixed_stable"
)


# Combine performance + trend
baseline["baseline_archetype"] = (
    baseline["performance_level"]
    + "_"
    + baseline["trend_level"]
)

In [ ]:
archetype_counts = (
    baseline["baseline_archetype"]
    .value_counts()
    .rename_axis("archetype")
    .reset_index(name="count")
)

archetype_counts["percentage"] = (
    archetype_counts["count"]
    / len(baseline)
    * 100
)

display(archetype_counts)

print("Total pages:", len(baseline))
print("Total classified:", archetype_counts["count"].sum())

,archetype,count,percentage
0,weak_mixed_stable,181673,71.299676
1,strong_improving,40539,15.910001
2,strong_declining,16681,6.546652
3,strong_mixed_stable,15671,6.150266
4,middle_improving,213,0.083594
5,middle_declining,24,0.009419
6,middle_mixed_stable,1,0.000392


Total pages: 254802
Total classified: 254802


In [ ]:
print(features_df["avg_monthly_impressions"].describe())

count    254802.000000
mean        516.881906
std        2426.619045
min           0.000000
25%           0.000000
50%           1.400000
75%         106.833333
max      152671.666667
Name: avg_monthly_impressions, dtype: float64


**Step 5**

In [ ]:
# ============================================================
# Step 5.1 — Inspect clustering features
# ============================================================

cluster_cols = [
    "avg_monthly_impressions",
    "avg_monthly_clicks",
    "avg_ctr",
    "avg_position",
    "impression_trend",
    "click_trend",
    "ctr_trend",
    "position_trend"
]

# Create a copy containing only the clustering features
X = features_df[cluster_cols].copy()

# Show basic statistics
display(
    X.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
    ).T
)

,count,mean,std,min,1%,25%,50%,75%,99%,max
avg_monthly_impressions,254802.0,516.881906,2426.619045,0.000000,0.000000,0.000000,1.400000,106.833333,8975.600000,152671.666667
avg_monthly_clicks,254802.0,1.679071,12.915675,0.000000,0.000000,0.000000,0.000000,0.166667,33.000000,1888.166667
avg_ctr,254802.0,0.002662,0.015045,0.000000,0.000000,0.000000,0.000000,0.000656,0.061235,0.466667
avg_position,155600.0,15.686423,16.544808,0.000000,1.000000,5.666667,9.396879,19.222420,77.000000,633.000000
impression_trend,254802.0,114.593870,796.065859,-19895.342857,-363.479143,0.000000,0.000000,19.514286,2483.254286,95443.085714
click_trend,254802.0,0.226623,3.727071,-195.342857,-1.942857,0.000000,0.000000,0.000000,6.314286,877.628571
ctr_trend,254802.0,-0.000123,0.009808,-0.300000,-0.011390,0.000000,0.000000,0.000000,0.007519,0.300000
position_trend,134869.0,0.379269,8.643601,-342.125000,-26.428571,-1.071429,0.175978,2.101190,24.396790,283.000000


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ============================================================
# Step 5.1.2 — Prepare clustering features
# ============================================================

cluster_cols = [
    "avg_monthly_impressions",
    "avg_monthly_clicks",
    "avg_ctr",
    "impression_trend",
    "click_trend",
    "ctr_trend"
]

X = features_df[cluster_cols].copy()

# ------------------------------------------------------------
# Log-transform highly skewed traffic variables
# ------------------------------------------------------------

X["avg_monthly_impressions"] = np.log1p(
    X["avg_monthly_impressions"]
)

X["avg_monthly_clicks"] = np.log1p(
    X["avg_monthly_clicks"]
)

# ------------------------------------------------------------
# Check for missing values
# ------------------------------------------------------------

print("Missing values:")
print(X.isna().sum())

# ------------------------------------------------------------
# Standardize all features
# ------------------------------------------------------------

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("\nOriginal shape:", X.shape)
print("Scaled shape:", X_scaled.shape)

print("\nScaled means:")
print(X_scaled.mean(axis=0))

print("\nScaled standard deviations:")
print(X_scaled.std(axis=0))

Missing values:
avg_monthly_impressions    0
avg_monthly_clicks         0
avg_ctr                    0
impression_trend           0
click_trend                0
ctr_trend                  0
dtype: int64

Original shape: (254802, 6)
Scaled shape: (254802, 6)

Scaled means:
[ 2.45843626e-16 -1.16452244e-16  1.69547328e-17  3.17901240e-17
  1.74566822e-17 -1.95202516e-18]

Scaled standard deviations:
[1. 1. 1. 1. 1. 1.]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd

# ============================================================
# Step 5.2 — Test different numbers of clusters
# ============================================================

# Use a representative sample for choosing K
sample_size = min(30000, len(X_scaled))

rng = np.random.RandomState(42)

sample_indices = rng.choice(
    len(X_scaled),
    size=sample_size,
    replace=False
)

X_sample = X_scaled[sample_indices]

print("Sample size:", len(X_sample))


# ------------------------------------------------------------
# Test K = 2 through 8
# ------------------------------------------------------------

results = []

for k in range(2, 9):

    print(f"Testing K = {k}...")

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_sample)

    inertia = kmeans.inertia_

    silhouette = silhouette_score(
        X_sample,
        labels
    )

    results.append({
        "k": k,
        "inertia": inertia,
        "silhouette_score": silhouette
    })


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

k_results = pd.DataFrame(results)

display(k_results)

Sample size: 30000
Testing K = 2...
Testing K = 3...
Testing K = 4...
Testing K = 5...
Testing K = 6...
Testing K = 7...
Testing K = 8...


,k,inertia,silhouette_score
0,2,162056.187473,0.659580
1,3,131530.601780,0.657220
2,4,108822.944418,0.671811
3,5,90065.887872,0.679709
4,6,73582.967320,0.668375
5,7,59775.233362,0.634737
6,8,51271.184480,0.641279


In [ ]:
from sklearn.cluster import KMeans

# ============================================================
# Step 5.3 — Final K-Means clustering
# ============================================================

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

# Assign every page to one of the 5 clusters
features_df["cluster"] = kmeans.fit_predict(X_scaled)

# ------------------------------------------------------------
# Check cluster sizes
# ------------------------------------------------------------

cluster_sizes = (
    features_df["cluster"]
    .value_counts()
    .sort_index()
)

print("Cluster sizes:")
display(cluster_sizes)

print("\nCluster percentages:")

cluster_percentages = (
    features_df["cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

display(cluster_percentages)

Cluster sizes:


,count
cluster,
0,210279
1,1203
2,41473
3,889
4,958



Cluster percentages:


,proportion
cluster,
0,82.53
1,0.47
2,16.28
3,0.35
4,0.38


In [ ]:
# ============================================================
# Step 5.4 — Profile the clusters
# ============================================================

cluster_profile = (
    features_df
    .groupby("cluster")[cluster_cols]
    .mean()
    .round(6)
)

display(cluster_profile)

,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend
cluster,,,,,,
0,34.076511,0.040491,0.001102,9.423252,0.006544,-0.000019
1,19981.136838,78.336672,0.004354,7568.071939,23.327788,-0.000117
2,2422.419611,7.820419,0.003933,436.629028,0.681107,-0.000305
3,34.091920,0.366310,0.147275,4.983554,-0.135528,-0.103600
4,4.535386,0.433699,0.153635,0.102967,0.185341,0.081109


In [ ]:
# ============================================================
# Step 5.5 — Inspect representative pages from each cluster
# ============================================================

for cluster_id in sorted(features_df["cluster"].unique()):
    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id}")
    print(f"{'='*70}")

    sample = (
        features_df[features_df["cluster"] == cluster_id]
        .sample(
            n=min(5, (features_df["cluster"] == cluster_id).sum()),
            random_state=42
        )
    )

    display(
        sample[
            [
                "content_hash_id",
                "avg_monthly_impressions",
                "avg_monthly_clicks",
                "avg_ctr",
                "impression_trend",
                "click_trend",
                "ctr_trend",
                "cluster"
            ]
        ]
    )


CLUSTER 0


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend,cluster
82269,content_6a14d55b4459656b,0.0,0.0,0.0,0.000000,0.0,0.0,0
133719,content_5cd5857aefa2feb0,0.0,0.0,0.0,0.000000,0.0,0.0,0
59392,content_ccc9cffd95f8fbc4,0.0,0.0,0.0,0.000000,0.0,0.0,0
228480,content_5b6b91455d4022f6,16.0,0.0,0.0,12.342857,0.0,0.0,0
96269,content_facc877792be2236,0.6,0.0,0.0,0.300000,0.0,0.0,0



CLUSTER 1


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend,cluster
234449,content_8d75e0387a0b4c23,12252.500000,1.333333,0.001068,8406.428571,0.628571,-0.000457,1
58264,content_55962d26ce96172b,10363.833333,44.666667,0.003578,4805.628571,22.114286,0.000534,1
6541,content_1b320fe45c240334,6326.833333,75.166667,0.014367,2240.657143,23.514286,-0.002556,1
154748,content_3e07fb36c28b980e,63867.666667,51.500000,0.000879,5408.171429,9.171429,0.000082,1
113563,content_6063ffe737ce7147,14605.000000,28.250000,0.002610,7309.400000,12.500000,-0.000882,1



CLUSTER 2


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend,cluster
4505,content_35db1b03de2ea628,642.833333,2.166667,0.006069,234.600000,0.200000,-0.002411,2
24595,content_03a79c8782deec41,598.666667,5.333333,0.009952,-160.742857,-1.028571,0.000245,2
50129,content_1032cde8d53cafde,1041.333333,0.500000,0.000465,676.514286,0.200000,0.000155,2
25884,content_4b48e60040f8b38f,1219.666667,2.833333,0.001669,299.428571,1.628571,0.000802,2
233252,content_84ac159666c9d8e3,15608.333333,84.833333,0.007534,5040.685714,-5.342857,-0.002037,2



CLUSTER 3


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend,cluster
91888,content_3121ca6171d7c2de,1.6,0.4,0.233333,-1.1,-0.2,-0.033333,3
124551,content_11d2ef31d1e3ba78,2.6,0.2,0.200000,1.4,-0.2,-0.200000,3
44442,content_551619948d8086a4,0.6,0.2,0.200000,0.1,-0.2,-0.200000,3
118541,content_87d1515de03c9761,0.4,0.2,0.200000,-0.1,-0.1,-0.100000,3
173193,content_4abc0677cb585cc0,10.6,0.4,0.083333,-0.5,-0.2,-0.066667,3



CLUSTER 4


,content_hash_id,avg_monthly_impressions,avg_monthly_clicks,avg_ctr,impression_trend,click_trend,ctr_trend,cluster
217184,content_cc98e7f8f652cf7d,3.8,0.4,0.095238,1.5,0.2,0.028571,4
130908,content_701f99e199bc6878,0.6,0.2,0.200000,0.5,0.1,0.100000,4
96272,content_443ab7186c2abc31,2.4,0.2,0.100000,-0.4,0.1,0.050000,4
242770,content_90c75f3c1b5349e2,0.6,0.2,0.100000,0.3,0.2,0.100000,4
241720,content_9a3d4fd4b579037b,6.8,0.6,0.102222,-0.8,0.3,0.060000,4
